# Camera Acquisition Application
This notebook implements a camera acquisition app using `PySpin` for camera control, `OpenCV` for image processing, and `Tkinter` for the GUI. It supports real-time video capture, annotation, and saving of raw video data.
### Imports
- `PySpin` for camera control
- `cv2` (OpenCV) for image processing and video saving
- `socket` for UDP communication
- `numpy` for numerical operations
- `csv` for CSV handling
- `datetime` for timestamps
- `tkinter` for GUI
- `threading` for concurrency
Ensure the `PySpin` SDK is installed from FLIR's website.

### `CameraApp` Class
Handles GUI and camera operations:
- **`__init__(self, root)`**: Initializes GUI components and sets up variables for video and CSV file handling.
- **`create_widgets(self)`**: Creates and arranges GUI elements for camera settings and control buttons.
- **`browse_output_file(self)`**: Allows user to select the output file path.
- **`configure_camera(self, cam)`**: Configures camera settings (mode, format, frame rate, exposure).
- **`acquire_video(self, cam)`**: Captures video frames, applies thresholding, and saves raw frames and annotations. 
- **`start_acquisition(self)`**: Starts acquisition in a separate thread.
- **`run_acquisition(self)`**: Manages the acquisition process, including camera initialization and frame processing.
- **`stop_acquisition_process(self)`**: Stops the acquisition process.

In [ ]:
import PySpin
import cv2
import socket
import numpy as np
import csv
from datetime import datetime
import tkinter as tk
from tkinter import ttk, filedialog
import threading
import queue

class CameraApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Camera Acquisition App")

        # Variables to store camera settings
        self.fps = tk.IntVar(value=50)
        self.exposure = tk.IntVar(value=5000)
        self.output_filename = tk.StringVar(value='output')
        self.min_threshold_value = tk.IntVar(value=200)
        self.max_threshold_value = tk.IntVar(value=255)
        self.min_blob_size = tk.IntVar(value=0)
        self.max_blob_size = tk.IntVar(value=300)
        self.save_video = tk.BooleanVar(value=True)

        # Create GUI elements
        self.create_widgets()

        # VideoWriter and CSV file variables
        self.out = None
        self.csv_file = None
        self.stop_acquisition = threading.Event()  # Event to signal stop of acquisition

    def create_widgets(self):
        # Frame for camera settings
        settings_frame = ttk.LabelFrame(self.root, text="Camera Settings")
        settings_frame.grid(row=0, column=0, padx=10, pady=10, sticky=tk.W)

        # FPS Entry
        ttk.Label(settings_frame, text="FPS:").grid(row=0, column=0, sticky=tk.W)
        fps_entry = ttk.Entry(settings_frame, textvariable=self.fps)
        fps_entry.grid(row=0, column=1, padx=5, pady=5)

        # Exposure Entry
        ttk.Label(settings_frame, text="Exposure (us):").grid(row=1, column=0, sticky=tk.W)
        exposure_entry = ttk.Entry(settings_frame, textvariable=self.exposure)
        exposure_entry.grid(row=1, column=1, padx=5, pady=5)

        # Output Filename Entry
        ttk.Label(settings_frame, text="Output Filename:").grid(row=2, column=0, sticky=tk.W)
        output_entry = ttk.Entry(settings_frame, textvariable=self.output_filename)
        output_entry.grid(row=2, column=1, padx=5, pady=5)

        # Min Threshold
        ttk.Label(settings_frame, text="Min Threshold:").grid(row=3, column=0, sticky=tk.W)
        self.min_threshold_value.get()  # Initialize value
        min_threshold_entry = ttk.Entry(settings_frame, textvariable=self.min_threshold_value)
        min_threshold_entry.grid(row=3, column=1, padx=5, pady=5)
        self.min_threshold_label = ttk.Label(settings_frame, text=str(self.min_threshold_value.get()))
        self.min_threshold_label.grid(row=3, column=2, padx=5, pady=5)
        min_threshold_scale = tk.Scale(settings_frame, from_=0, to=255, orient=tk.HORIZONTAL, variable=self.min_threshold_value, command=self.update_min_threshold)
        min_threshold_scale.grid(row=3, column=3, padx=5, pady=5)

        # Max Threshold
        ttk.Label(settings_frame, text="Max Threshold:").grid(row=4, column=0, sticky=tk.W)
        self.max_threshold_value.get()  # Initialize value
        max_threshold_entry = ttk.Entry(settings_frame, textvariable=self.max_threshold_value)
        max_threshold_entry.grid(row=4, column=1, padx=5, pady=5)
        self.max_threshold_label = ttk.Label(settings_frame, text=str(self.max_threshold_value.get()))
        self.max_threshold_label.grid(row=4, column=2, padx=5, pady=5)
        max_threshold_scale = tk.Scale(settings_frame, from_=0, to=255, orient=tk.HORIZONTAL, variable=self.max_threshold_value, command=self.update_max_threshold)
        max_threshold_scale.grid(row=4, column=3, padx=5, pady=5)

        # Min Blob Size
        ttk.Label(settings_frame, text="Min Blob Size:").grid(row=5, column=0, sticky=tk.W)
        self.min_blob_size.get()  # Initialize value
        min_blob_size_entry = ttk.Entry(settings_frame, textvariable=self.min_blob_size)
        min_blob_size_entry.grid(row=5, column=1, padx=5, pady=5)
        self.min_blob_size_label = ttk.Label(settings_frame, text=str(self.min_blob_size.get()))
        self.min_blob_size_label.grid(row=5, column=2, padx=5, pady=5)
        min_blob_size_scale = tk.Scale(settings_frame, from_=0, to=2000, orient=tk.HORIZONTAL, variable=self.min_blob_size, command=self.update_min_blob_size)
        min_blob_size_scale.grid(row=5, column=3, padx=5, pady=5)

        # Max Blob Size
        ttk.Label(settings_frame, text="Max Blob Size:").grid(row=6, column=0, sticky=tk.W)
        self.max_blob_size.get()  # Initialize value
        max_blob_size_entry = ttk.Entry(settings_frame, textvariable=self.max_blob_size)
        max_blob_size_entry.grid(row=6, column=1, padx=5, pady=5)
        self.max_blob_size_label = ttk.Label(settings_frame, text=str(self.max_blob_size.get()))
        self.max_blob_size_label.grid(row=6, column=2, padx=5, pady=5)
        max_blob_size_scale = tk.Scale(settings_frame, from_=0, to=2000, orient=tk.HORIZONTAL, variable=self.max_blob_size, command=self.update_max_blob_size)
        max_blob_size_scale.grid(row=6, column=3, padx=5, pady=5)

        # Checkbox for video saving
        save_video_checkbox = ttk.Checkbutton(settings_frame, text="Save Video", variable=self.save_video)
        save_video_checkbox.grid(row=7, column=0, columnspan=4, padx=5, pady=5)

        # Start Button
        start_button = ttk.Button(self.root, text="Start Acquisition", command=self.start_acquisition)
        start_button.grid(row=1, column=0, padx=10, pady=10)

        # Stop Button
        stop_button = ttk.Button(self.root, text="Stop Acquisition", command=self.stop_acquisition_process)
        stop_button.grid(row=1, column=1, padx=10, pady=10)

    def update_min_threshold(self, value):
        self.min_threshold_label.config(text=value)
        self.min_threshold_value.set(value)  # Ensure entry field is updated

    def update_max_threshold(self, value):
        self.max_threshold_label.config(text=value)
        self.max_threshold_value.set(value)  # Ensure entry field is updated

    def update_min_blob_size(self, value):
        self.min_blob_size_label.config(text=value)
        self.min_blob_size.set(value)  # Ensure entry field is updated

    def update_max_blob_size(self, value):
        self.max_blob_size_label.config(text=value)
        self.max_blob_size.set(value)  # Ensure entry field is updated



    def browse_output_file(self):
        filename = filedialog.asksaveasfilename(defaultextension=".avi", filetypes=[("AVI files", "*.avi")])
        if filename:
            self.output_filename.set(filename)

    def configure_camera(self, cam):
        nodemap = cam.GetNodeMap()
        
        # Set acquisition mode to continuous
        node_acquisition_mode = PySpin.CEnumerationPtr(nodemap.GetNode('AcquisitionMode'))
        node_acquisition_mode_continuous = node_acquisition_mode.GetEntryByName('Continuous')
        node_acquisition_mode.SetIntValue(node_acquisition_mode_continuous.GetValue())
        
        # Set pixel format to Mono8 (8-bit grayscale)
        node_pixel_format = PySpin.CEnumerationPtr(nodemap.GetNode('PixelFormat'))
        node_pixel_format_mono8 = node_pixel_format.GetEntryByName('Mono8')
        node_pixel_format.SetIntValue(node_pixel_format_mono8.GetValue())
        
        # Set maximum width
        node_width = PySpin.CIntegerPtr(nodemap.GetNode('Width'))
        max_width = node_width.GetMax()
        node_width.SetValue(max_width)
        
        # Set maximum height
        node_height = PySpin.CIntegerPtr(nodemap.GetNode('Height'))
        max_height = node_height.GetMax()
        node_height.SetValue(max_height)
        
        # Enable frame rate control
        node_acquisition_frame_rate_enable = PySpin.CBooleanPtr(nodemap.GetNode('AcquisitionFrameRateEnable'))
        node_acquisition_frame_rate_enable.SetValue(True)

        # Set frame rate
        node_frame_rate = PySpin.CFloatPtr(nodemap.GetNode('AcquisitionFrameRate'))
        node_frame_rate.SetValue(self.fps.get())
        
        # Turn off auto exposure
        node_exposure_auto = PySpin.CEnumerationPtr(nodemap.GetNode('ExposureAuto'))
        node_exposure_auto_off = node_exposure_auto.GetEntryByName('Off')
        node_exposure_auto.SetIntValue(node_exposure_auto_off.GetValue())
        
        # Turn off auto gain
        node_gain_auto = PySpin.CEnumerationPtr(nodemap.GetNode('GainAuto'))
        node_gain_auto_off = node_gain_auto.GetEntryByName('Off')
        node_gain_auto.SetIntValue(node_gain_auto_off.GetValue())
        
        # Set exposure time (in microseconds)
        node_exposure_time = PySpin.CFloatPtr(nodemap.GetNode('ExposureTime'))
        node_exposure_time.SetValue(self.exposure.get())
        
        # Set gain (if needed)
        node_gain = PySpin.CFloatPtr(nodemap.GetNode('Gain'))
        node_gain.SetValue(10.0)  # Example: set gain to 10 dB

    def acquire_video(self, cam):
        # Set up socket for UDP communication
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        serverAddressPort = ("127.0.0.1", 5053)

        # Initialize CSV writer
        self.csv_file = open(f'{self.output_filename.get()}.csv', 'w', newline='')
        csv_writer = csv.writer(self.csv_file)
        csv_writer.writerow(['Timestamp', 'Center_X_mm', 'Center_Y_mm'])

        # Initialize VideoWriter if "Save Video" checkbox is checked
        if self.save_video.get():
            fourcc = cv2.VideoWriter_fourcc(*'MJPG')
            frame_width = int(cam.Width())
            frame_height = int(cam.Height())
            self.out = cv2.VideoWriter(f'{self.output_filename.get()}.avi', fourcc, self.fps.get(), (frame_width, frame_height), isColor=False)
        else:
            self.out = None

        try:
            # Start the acquisition
            cam.BeginAcquisition()

            while not self.stop_acquisition.is_set():
                # Retrieve next image
                image_result = cam.GetNextImage()

                if image_result.IsIncomplete():
                    print('Image incomplete: ', image_result.GetImageStatus())
                else:
                    # Convert image to numpy array (Grayscale)
                    frame = image_result.GetNDArray()

                    # Create a copy of the frame for annotation
                    annotated_frame = frame.copy()

                    # Apply binary thresholding to create a mask
                    _, mask = cv2.threshold(frame, self.min_threshold_value.get(), self.max_threshold_value.get(), cv2.THRESH_BINARY)

                    # Find contours in the binary mask
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                    # Check if any contours are found
                    if contours:
                        for contour in contours:
                            area = cv2.contourArea(contour)
                            if self.min_blob_size.get() <= area <= self.max_blob_size.get():
                                M = cv2.moments(contour)
                                if M["m00"] != 0:
                                    cX = int(M["m10"] / M["m00"])
                                    cY = int(M["m01"] / M["m00"])

                                    # Convert pixel coordinates to millimeters from the center
                                    x_mm = (cX - frame.shape[1] / 2) * (300 / 1280)
                                    y_mm = (frame.shape[0] / 2 - cY) * (300 / 1280)

                                    # Get current timestamp
                                    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

                                    # Write data to CSV file
                                    csv_writer.writerow([timestamp, x_mm, y_mm])
                                    self.csv_file.flush()  # Ensure data is written immediately

                                    data = (y_mm, -x_mm)      # Changed this to rotate it -90 degrees 
                                    print(data)
                                    sock.sendto(str.encode(str(data)), serverAddressPort)

                                    # Draw the bounding rectangle and center point on the annotated frame
                                    (x, y, w, h) = cv2.boundingRect(contour)
                                    cv2.rectangle(annotated_frame, (x, y), (x+w, y+h), (255, 255, 255), 2)  # White rectangle
                                    cv2.putText(annotated_frame, f"Area: {int(area)}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

                    # Display the annotated frame
                    annotated_frame_resized = cv2.resize(annotated_frame, (0, 0), None, 0.5, 0.5)

                    # Add crosshair to the center of the frame
                    height, width = annotated_frame_resized.shape[:2]
                    center_x, center_y = width // 2, height // 2
                    cv2.line(annotated_frame_resized, (0, center_y), (width, center_y), (255, 255, 255), 1)  # Horizontal line
                    cv2.line(annotated_frame_resized, (center_x, 0), (center_x, height), (255, 255, 255), 1)  # Vertical line
                    
                    cv2.imshow('Video', annotated_frame_resized)

                    # Save the raw frame to the video file
                    if self.out is not None:
                        self.out.write(frame)

                    # Check for 'Esc' key press to exit (optional)
                    if cv2.waitKey(1) == 27:  # Esc key
                        break

                # Release image
                image_result.Release()

        finally:
            # End acquisition
            cam.EndAcquisition()

            # Close CSV file
            self.csv_file.close()

            # Release VideoWriter
            if self.out is not None:
                self.out.release()

            # Close socket
            sock.close()

            # Destroy OpenCV window
            cv2.destroyAllWindows()

    def start_acquisition(self):
        # Start acquisition in a separate thread
        self.stop_acquisition.clear()  # Clear the stop event flag
        acquisition_thread = threading.Thread(target=self.run_acquisition)
        acquisition_thread.start()

    def run_acquisition(self):
        system = PySpin.System.GetInstance()
        cam_list = system.GetCameras()

        try:
            if cam_list.GetSize() == 0:
                raise ValueError("No cameras found!")

            # Assume one camera for simplicity
            cam = cam_list.GetByIndex(0)

            # Initialize camera
            cam.Init()

            # Configure camera settings
            self.configure_camera(cam)

            # Acquire video until stop event is set
            self.acquire_video(cam)

            # Deinitialize camera
            cam.DeInit()

        except PySpin.SpinnakerException as ex:
            print('Error: %s' % ex)
            exit_code = 1

        finally:
            # Release camera and system resources
            del cam
            cam_list.Clear()
            system.ReleaseInstance()

    def stop_acquisition_process(self):
        # Set the stop event flag to stop acquisition
        self.stop_acquisition.set()

if __name__ == '__main__':
    root = tk.Tk()
    app = CameraApp(root)
    root.mainloop()